# Домашнее задание
I. ответить на вопросы

1.   Как определить границы применимости модели?    Почему в оптимизатор бюджета можно закладывать новый бюджет, отличающийся не более 20% от реального?
2.   Если какой-то канал слабее -- почему вообще не прекратить им пользоваться?
3.   Как учесть ограничения от клиента -- например, не более 30% на tv? или при бюджете на internet не более 100 000 долларов?
4.   Что произойдет с моделью если в данных будут пропуски по каким-то каналам?
5.   Что произойдет с моделью если все рекламные кампании в прошлом имели одинаковый медиа микс?

II. Модель на реальном (условно) датасете в нашем примере получилась здорово переобученной (посмотрите как падает R^2)
1.   Переобучите модель (можно взять любую библиотеку / написать самим) так чтобы R^2 на том же тесте был не менее 0.65 и при этом сохранилась интерпретабельность коэффициентов.
2.   Посчитайте рекомендуемый медиамикс для того же бюджета по новой модели и сравните с тем что получилось

1. Кривые оценивающие вклад канала могут выглядеть по-другому, если бюджет сильно отличается, так как модели не кормили данные с таким бюдетом, поэтому применимость может быть ограничена размахом бюджета. Также в меминаре говорили про то, что инфляция может сильно влять на денежные факторы, возможно это тоже ограничивает применимость устаревшей модели в новых экономических условиях.
На S-образных кривых дохода от каналов доверительные интервалы будут большими, если отойти от 20% и можно сильно ошибиться, к тому же вряд ли компания благодаря твоей модели будет резко менять бюджет.
2. Из-за структуры взаимодействия с каналом, чем меньше им пользоваться, тем больше выручки принесет увеличение на условную единицу расходов на него, поэтому можно слегка снизить бюджет на какой-то канал и спуститься по кривой, но дальше спуск станет еще круче.
3. В Meridian можно задать ограничения напрямую, но также можно решать эту задачу как задачу линейного программирования, это только что у нас было на курсе по методам оптимизации, поэтому у меня руки чешуться применить куда-то ADMM :) Но для похожей структуры надо будет вручную программировать дополнительные переменные типа тренда и разложение для сезонности и отдельно встраивать байесы, так что лучше просто использовать Meridian.
4. Можно заполнить разными значениями в зависимости от задачи, например предыдущим значением. Если много пропусков, то можно удалить канал из анализа.
5. Есть подозрение, что у нас могут не сойтись коэффициенты, либо мы получим высокую неопределенность в прогнозах.

У меня не запускается ваш код с семинара никаким образом. Я потратил уже больше пяти часов на удаление/скачивание разных версий пакетов и у меня так и не запустился код с семинара. В колабе он также не работает, поэтому я обучу просто линейную регрессию, но этого конечно не хватит даже чтобы побить переобученный тест :(

In [1]:
!wget https://github.com/MLinside-team/ML-in-business/raw/refs/heads/main/datasets/MMM_data.xlsx

--2025-06-18 01:01:07--  https://github.com/MLinside-team/ML-in-business/raw/refs/heads/main/datasets/MMM_data.xlsx
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MLinside-team/ML-in-business/refs/heads/main/datasets/MMM_data.xlsx [following]
--2025-06-18 01:01:07--  https://raw.githubusercontent.com/MLinside-team/ML-in-business/refs/heads/main/datasets/MMM_data.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 395020 (386K) [application/octet-stream]
Saving to: ‘MMM_data.xlsx.1’

MMM_data.xlsx.1     100%[===================>] 385.76K  2.48MB/s    in 0.2s    

2025-06-18 01:01:08 (2.48 MB/s)

In [2]:
import pandas as pd
real_data = pd.read_excel('MMM_data.xlsx')
real_data.head()

,TV Manufacturing Brand,DATE,DEMAND,Consumer Price Index (CPI),Consumer Confidence Index(CCI),Producer Price Index (PPI),Unit Price ($),POS/ Supply Data,SALES ($),Advertising Expenses (SMS),Advertising Expenses(Newspaper ads),Advertising Expenses(Radio),Advertising Expenses(TV),Advertising Expenses(Internet),GRP (NewPaper ads),GRP(SMS),GRP(Radio,GRP(Internet),GRP(TV)
0,TV Manufacturing & Supplier Unit,2010-01-01,4384,104.9,96.3,106.7,361.62,4240,1533268.80,77.4819,14.104193,112.3370,1479.4565,722.571,95.333,11.8398,91.0000,276.3636,756.5909
1,TV Manufacturing & Supplier Unit,2010-01-02,4366,104.9,96.3,106.7,361.62,4266,1542670.92,73.4783,13.298758,105.7133,1369.8913,717.857,114.957,27.8039,111.9091,291.3182,860.1364
2,TV Manufacturing & Supplier Unit,2010-01-03,4006,104.9,96.3,106.7,361.62,4206,1520973.72,80.6093,13.200691,108.7702,1428.0645,653.333,113.090,0.0000,94.6364,282.7273,751.9545
3,TV Manufacturing & Supplier Unit,2010-01-04,4076,104.9,96.3,106.7,361.62,4176,1510125.12,65.9319,12.721429,93.3065,1309.3548,622.095,72.442,0.0000,98.5909,306.3182,749.4545
4,TV Manufacturing & Supplier Unit,2010-01-05,4834,104.9,96.3,106.7,361.62,5234,1892719.08,77.4819,14.104193,112.3370,1479.4565,722.571,95.333,11.8398,91.0000,276.3636,756.5909


In [8]:
for col in real_data.columns:
  print(f"'{col}'")

'TV Manufacturing Brand'
'DATE'
'DEMAND '
'Consumer Price Index (CPI)'
'Consumer Confidence Index(CCI)'
'Producer Price Index (PPI)'
'Unit Price ($)'
'POS/ Supply Data'
'SALES ($)'
'Advertising Expenses (SMS)'
'Advertising Expenses(Newspaper ads)'
'Advertising Expenses(Radio)'
'Advertising Expenses(TV)'
'Advertising Expenses(Internet)'
'GRP (NewPaper ads)'
'GRP(SMS)'
'GRP(Radio'
'GRP(Internet)'
'GRP(TV)'


In [12]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

data = pd.read_excel('MMM_data.xlsx')

X = data.drop([
    'TV Manufacturing Brand', 'DATE', 'SALES ($)', 
    'DEMAND ', 'POS/ Supply Data', 'Unit Price ($)'
], axis=1)
y = np.log(data['SALES ($)'])


ad_cols = [c for c in X.columns if 'Advertising' in c]
for col in ad_cols:
    X[col] = np.sqrt(X[col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Ridge(alpha=1.0, random_state=42)
model.fit(X_train_scaled, y_train)

train_score = r2_score(y_train, model.predict(X_train_scaled))
test_score = r2_score(y_test, model.predict(X_test_scaled))

print(f"R² (train): {train_score:.4f}")
print(f"R² (test): {test_score:.4f}")

R² (train): 0.2147
R² (test): 0.2211
